# Pandas vs PySpark

The pandas operations you already know (`head`, `info`, `value_counts`, filter, `groupby`, `merge`), side by side with the Spark version.

Data: Superstore, exported as 3 CSV files in `/workspace/data/`:

| File | Content |
| --- | --- |
| `superstore_orders.csv` | one row per order line: dates, customer ID, product ID, location, sales |
| `superstore_customers.csv` | customer ID, name, segment |
| `superstore_products.csv` | product ID, name, category |

Next notebook: `02_etl_superstore.ipynb` uses these operations to build an ETL pipeline.

### What is Spark?
- pandas works on **one computer** and keeps all data in memory.
- Spark splits the data into **partitions** and processes them **in parallel**, so it can handle data that is too big for pandas.
- The code looks very similar to pandas. The two biggest differences:
  1. Spark is **lazy**: it only computes when you ask for a result (`show`, `count`, `write`).
  2. A Spark DataFrame **cannot be changed in place**. Always save the result: `df = df.withColumn(...)`.

### SETUP

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("pandas-vs-spark").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")   # hide Spark warning messages
spark

- The first run prints a few `WARNING` lines while Spark starts. That is normal.
- `SparkSession` is the entry point to Spark, like `import pandas as pd`.
- `functions as F` holds Spark's column functions (`F.col`, `F.sum`, `F.trim`, ...).
- While the session is running, open the Spark UI at http://localhost:4040.

## Create DataFrame

pandas: `pd.DataFrame(data)` → Spark: `spark.createDataFrame(...)`. A pandas DataFrame can be converted directly.

In [ ]:
data = {'Name': ['Alice', 'Bob', 'Charlie'],
        'Age': [25, 30, 22],
        'City': ['New York', 'London', 'Paris']}

pdf = pd.DataFrame(data)          # pandas
sdf = spark.createDataFrame(pdf)  # Spark
sdf.show()

## Loading Data

pandas: `pd.read_csv('file.csv')` → Spark: `spark.read.csv('file.csv', header=True, inferSchema=True, escape='"')`

- `header=True`: the first row is the column names (pandas does this by default).
- `inferSchema=True`: let Spark detect the data types (pandas does this by default).
- `escape='"'`: some text contains commas and quotes, for example `Chair, "Executive Red"`.
  pandas handles this automatically; Spark needs this option, otherwise the columns shift.
- The path starts with `/workspace/...` because the notebook runs inside the Docker container.

In [ ]:
orders = spark.read.csv('/workspace/data/superstore_orders.csv', header=True, inferSchema=True, escape='"')

## Simple Exploring Data

**Show first 5 rows** → pandas: `df.head()`

In [ ]:
orders.show(5)

Too wide? Convert a few rows to pandas to get the familiar table:

In [ ]:
orders.limit(5).toPandas()

**Number of rows and columns** → pandas: `df.shape`

In [ ]:
(orders.count(), len(orders.columns))

**DataFrame summary** → pandas: `df.info()`

In [ ]:
orders.printSchema()

**Descriptive statistics** → pandas: `df.describe()`

In [ ]:
orders.describe('Sales', 'Quantity', 'Discount', 'Profit').show()

`Sales` has fewer values than the other columns, so some values are missing.

**Column list and data types** → pandas: `df.columns`, `df.dtypes`

In [ ]:
orders.columns

In [ ]:
orders.dtypes

**Number of unique values** → pandas: `df.nunique()`

In [ ]:
orders.select(F.countDistinct('Order ID'), F.countDistinct('Customer ID'), F.countDistinct('Product ID')).show()

**Frequency count** → pandas: `df['Region'].value_counts()`

In [ ]:
orders.groupBy('Region').count().orderBy(F.desc('count')).show()

`WEST` and `West` are the same region written differently. This dirty data is cleaned in the ETL notebook.

**Unique values** → pandas: `df['Ship Mode'].unique()`

In [ ]:
orders.select('Ship Mode').distinct().show()

## Basic Data Manipulation

**Sorting** → pandas: `df.sort_values(by='Sales', ascending=False)`

In [ ]:
orders.orderBy(F.desc('Sales')).select('Order ID', 'Product ID', 'Sales').show(5)

**Accessing columns** → pandas: `df[['Order ID', 'Sales']]`

In [ ]:
orders.select('Order ID', 'Sales').show(3)

**Filtering, single condition** → pandas: `df[df['Sales'] > 3000]`

In [ ]:
orders.filter(F.col('Sales') > 3000).select('Order ID', 'State', 'Sales').show()

**Filtering, multiple conditions** → pandas: `df[(df['Sales'] > 1000) & (df['Region'] == 'West')]`

In [ ]:
orders.filter((F.col('Sales') > 1000) & (F.col('Region') == 'West')).select('Order ID', 'Region', 'Sales').show(5)

**Query string** → pandas: `df.query('Sales > 3000')`

In [ ]:
orders.filter('Sales > 3000').count()

**isin** → pandas: `df[df['State'].isin(['California', 'Texas'])]`

In [ ]:
orders.filter(F.col('State').isin('California', 'Texas')).groupBy('State').count().show()

**Text contains** → pandas: `df[df['City'].str.contains('San')]`

In [ ]:
orders.filter(F.col('City').contains('San')).select('City').distinct().show()

**Add a new column** → pandas: `df['Unit Price'] = df['Sales'] / df['Quantity']`

Spark cannot change a DataFrame in place, so the result must be saved to a variable.

In [ ]:
with_price = orders.withColumn('Unit Price', F.round(F.col('Sales') / F.col('Quantity'), 2))
with_price.select('Sales', 'Quantity', 'Unit Price').show(3)

**Remove a column** → pandas: `df.drop(columns='Unit Price')`

In [ ]:
with_price.drop('Unit Price').columns[-4:]

**Grouping and aggregation** → pandas: `df.groupby('Region').agg({'Sales': 'sum', 'Order ID': 'count'})`

In [ ]:
(orders.groupBy('Region')
       .agg(F.round(F.sum('Sales'), 2).alias('Total Sales'),
            F.count('Order ID').alias('Rows'))
       .orderBy(F.desc('Total Sales'))
       .show())

## Combining Data from Several Files (join)

`orders` only has `Product ID` and `Customer ID`. The product name, category and customer name are in the other files.

pandas: `pd.merge(orders, products, on='Product ID', how='left')` → Spark: `orders.join(products, on='Product ID', how='left')`

In [ ]:
products = spark.read.csv('/workspace/data/superstore_products.csv', header=True, inferSchema=True, escape='"')
customers = spark.read.csv('/workspace/data/superstore_customers.csv', header=True, inferSchema=True, escape='"')

products.show(3, truncate=False)
customers.show(3)

**Join 2 tables**: add product information to each order

In [ ]:
orders_products = orders.join(products, on='Product ID', how='left')
orders_products.select('Order ID', 'Product ID', 'Product Name', 'Category', 'Sales').show(5, truncate=False)

Now we can group by a column that only exists in the products file. `NULL` comes from orders whose product is missing in the products file (see `left_anti` below):

In [ ]:
orders_products.groupBy('Category').agg(F.round(F.sum('Sales'), 2).alias('Total Sales')).show()

**Join 3 tables**: chain `.join()` like chaining `pd.merge()`.

Watch the row count: `orders` has 1,436 rows, but the result has more. The customers and products files contain
**duplicate IDs**, and a join repeats an order once for every matching row. Always check the row count after a join.
The ETL notebook removes the duplicates before joining.

In [ ]:
full = (orders
        .join(customers, on='Customer ID', how='left')
        .join(products, on='Product ID', how='left'))

print('orders:', orders.count(), '| after join:', full.count())
full.select('Order ID', 'Customer Name', 'Segment', 'Product Name', 'Sales').show(5, truncate=False)

**Join types** (same as pandas `how=` and SQL):

| `how=` | Keeps |
| --- | --- |
| `'inner'` | only rows that match in both tables |
| `'left'` | all rows of the left table |
| `'right'` | all rows of the right table |
| `'outer'` | all rows of both tables |
| `'left_anti'` | rows of the left table **without** a match (useful to find missing data) |

In [ ]:
orders.join(products, on='Product ID', how='left_anti').select('Order ID', 'Product ID').show()

These orders refer to products that are not in the products file. The ETL notebook handles them.

## Pandas vs Spark summary

| pandas | Spark |
| --- | --- |
| `pd.read_csv(path)` | `spark.read.csv(path, header=True, inferSchema=True, escape='"')` |
| `df.head()` | `df.show(5)` |
| `df.shape` | `(df.count(), len(df.columns))` |
| `df.info()` | `df.printSchema()` |
| `df['col'].value_counts()` | `df.groupBy('col').count()` |
| `df['col'].unique()` | `df.select('col').distinct()` |
| `df.sort_values('col')` | `df.orderBy('col')` |
| `df[['a', 'b']]` | `df.select('a', 'b')` |
| `df[df['a'] > 1]` | `df.filter(F.col('a') > 1)` |
| `df['new'] = ...` | `df = df.withColumn('new', ...)` |
| `df.drop(columns='a')` | `df.drop('a')` |
| `df.groupby('a').agg(...)` | `df.groupBy('a').agg(...)` |
| `pd.merge(a, b, on='id', how='left')` | `a.join(b, on='id', how='left')` |
| (already pandas) | `df.toPandas()` |

Done. Stop Spark to free the laptop's memory.

In [ ]:
spark.stop()